#### #192 Конверсия поиска в просмотр: с фильтрами и без

Продуктовая гипотеза: запросы с фильтрами чаще приводят к просмотру объявления. Проверьте её — посчитайте конверсию поиска в просмотр отдельно для запросов с фильтрами и без.

In [ ]:
SELECT 
    s.used_filters
    ,COUNT(DISTINCT s.search_id) AS searches_count
    ,ROUND(
        COUNT(DISTINCT lv.search_id) / COUNT(DISTINCT s.search_id)::numeric * 100
        , 2
    ) as conversion_rate
FROM searches s
LEFT JOIN listing_views lv ON lv.search_id = s.search_id
GROUP BY s.used_filters

#### #93 Средний возраст покупателей Smartwatch

Какой средний возраст клиентов, купивших Smartwatch в 2024 году?

In [ ]:
SELECT 
    AVG(age) as average_age
FROM (
    SELECT DISTINCT 
        c.customer_key
        ,c.age
    FROM Customer c 
    JOIN Purchase pu ON c.customer_key = pu.customer_key
    JOIN Product pr ON pr.product_key = pu.product_key
    WHERE pr.name = 'Smartwatch'
        AND EXTRACT(YEAR FROM pu.date) = 2024
) 

#### #94 Покупатели Laptop и Monitor в марте

Вывести имена покупателей, каждый из которых приобрёл Laptop и Monitor в марте 2024 года

In [ ]:
SELECT
    c.name
FROM Customer c 
JOIN Purchase pu ON pu.customer_key = c.customer_key
JOIN Product pr ON pr.product_key = pu.product_key
WHERE (pr.name = 'Laptop' OR pr.name = 'Monitor')
    AND EXTRACT(MONTH FROM pu.date) = 3
    AND EXTRACT(YEAR FROM pu.date) = 2024
GROUP BY c.customer_key 
HAVING COUNT(DISTINCT pr.name) = 2

#### #97 Работающие склады по городам

Посчитать количество работающих складов на текущую дату по каждому городу. Вывести только те города, у которых количество складов более 80.

In [ ]:
SELECT 
    city
    ,COUNT(city) AS warehouse_count
FROM Warehouses
WHERE date_close IS NULL
GROUP BY city
HAVING COUNT(city) > 80

#### #101 Первый заказ пользователя

Выведи для каждого пользователя первое наименование, которое он заказал (первое по времени транзакции).

In [ ]:
SELECT 
    user_id
    ,item
FROM ( 
    SELECT 
        user_id
        ,item
        ,DENSE_RANK() OVER(PARTITION BY user_id ORDER BY transaction_ts ASC) AS dense_rank
    FROM Transactions
)
WHERE dense_rank = 1

#### #111 Население каждого региона

Посчитайте население каждого региона.

In [ ]:
SELECT 
    r.name as region_name
    ,SUM(c.population) as total_population
FROM Regions r 
JOIN Cities c ON c.regionid = r.id
GROUP BY r.name

#### #123 Количество задач у сотрудника

Необходимо написать SQL-запрос, который покажет всех сотрудников, у кого в работе менее трех задач

In [ ]:
SELECT 
    e.emp_name
    ,COUNT(t.assignee_id) as task_count
FROM Employee e
JOIN Tasks t ON t.assignee_id = e.id
GROUP BY e.id
HAVING COUNT(t.assignee_id) < 3

#### #125 Произведения, издававшиеся более 5 раз

Найти произведения, которые издавались более 5 раз. 

In [ ]:
SELECT 
    b.title
FROM Books b
JOIN BookEditions be ON be.book_id = b.id
GROUP BY b.title
HAVING COUNT(be.publish_year) > 5

#### #129 Номера счетов компаний с SBER в названии

Найти номера счетов по компаниям, в названии которых встречается 'SBER'.

In [ ]:
SELECT 
    contr.contract_number
FROM Contract contr
JOIN Company_contract cc ON cc.contract_id = contr.contract_id
JOIN Company comp on comp.company_id = cc.company_id
WHERE comp.company_name LIKE '%SBER%'

#### #131 Количество заказов каждого зарегистрированного клиента

Напишите запрос для определения количества заказов, размещенных каждым зарегистрированным клиентом.

In [ ]:
SELECT 
    c.customer_id
    ,COUNT(p.customer_id) as total_orders
FROM registered_customers c
LEFT JOIN purchases p ON c.customer_id = p.customer_id
GROUP BY c.customer_id

#### #133 Проекты, которые никогда не брались в работу

Вывести все проекты, которые никогда не брались в работу разработчиками.

In [ ]:
SELECT 
    p.name
FROM Projects p
LEFT JOIN ProjectHistory ph ON ph.project_id = p.id
WHERE ph.start_date IS NULL

#### #153 Самый активный клиент-рецензент

Напишите запрос, который выведет имя клиента с наибольшим числом оставленных оценок.

In [ ]:
SELECT 
    c.name
FROM EngagementRatings er
JOIN Customers c ON c.customer_id = er.customer_id
GROUP BY c.name
ORDER BY COUNT(*) DESC
LIMIT 1

#### #156 Средний рейтинг товара по месяцам

Напишите запрос, который для каждой пары (товар, месяц) вернёт product_id, номер месяца mth и средний рейтинг avg_stars, округлённый до двух знаков после запятой. Учитывайте только товары, по которым были отзывы.

In [ ]:
SELECT 
    product_id
    ,TO_CHAR(submit_date, 'MM')::integer  AS mth
    ,ROUND(
        AVG(stars)
        , 2
    ) AS avg_stars
FROM reviews
GROUP BY product_id, TO_CHAR(submit_date, 'MM')

#### #157 Среднее число позиций в заказе

Напишите запрос, который вернёт одно число — среднее количество товаров в одном заказе по платформе

In [ ]:
SELECT
    ROUND(
        AVG(quantity_per_order)
        , 2
    ) AS avg_items_per_order
FROM (
    SELECT 
        SUM(quantity) AS quantity_per_order
    FROM order_items
    GROUP BY order_id
)

####  #160 Топ-5 покупателей за последние 30 дней

Условно сегодняшняя дата — 31 марта 2024 года. Считайте «последними 30 днями» интервал с 2 марта 2024 по 31 марта 2024 включительно.

Напишите запрос, который вернёт customer_name и order_count пяти покупателей с наибольшим числом заказов за этот период. Тестовые данные исключают одинаковое число заказов у разных покупателей внутри топ-5

In [ ]:
SELECT 
    c.name AS customer_name
    ,COUNT(o.customer_id) AS order_count
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.order_delivered > '2024-03-02'
    AND o.order_delivered < '2024-03-31'
GROUP BY c.name
ORDER BY order_count DESC 
LIMIT 5

####  #167 Активные годовые подписки

Напишите запрос, который вернёт customer_name и product_name для всех клиентов с активной на сегодня подпиской типа annual. Подписка активна, если CURRENT_DATE попадает в интервал [start_date, end_date].

In [ ]:
SELECT 
    c.name AS customer_name
    ,s.product_name
FROM subscriptions s 
JOIN customers c ON c.customer_id = s.customer_id
WHERE subscription_type = 'annual'
    AND CURRENT_DATE BETWEEN s.start_date AND s.end_date
ORDER BY customer_name

####  #161 Среднее время доставки по городам

Напишите запрос, который вернёт название города и среднее время доставки по каждому городу

In [ ]:
SELECT
    c.city_name,
    ROUND(
        AVG(
            EXTRACT(EPOCH FROM (o.order_delivered - o.order_placed)) / 60
        )
    , 2) AS avg_minutes
FROM orders o
JOIN restaurants r ON r.restaurant_id = o.restaurant_id
JOIN cities c ON c.city_id = r.city_id
GROUP BY c.city_name
ORDER BY avg_minutes

#### #169 Топ-10 PC-игроков по сумме ставок

Возьмите только те ставки bets, которые сделаны на играх с категорией PC. 
Напишите запрос, который вернёт топ-10 игроков по суммарной сумме таких ставок. Для каждого игрока выведите его имя, число уникальных PC-игр, на которые он ставил и сумму ставок.

In [ ]:
SELECT  
    c.name as customer_name
    ,COUNT(DISTINCT g.game_id) as pc_games_played
    ,SUM(b.bet_amount) as total_spend
FROM bets b
JOIN games g ON g.game_id = b.game_id
JOIN customers c ON c.customer_id = b.customer_id
WHERE g.game_category = 'PC'
GROUP BY c.name
ORDER BY total_spend DESC 
LIMIT 10

#### #172 Расходы по категориям мерчантов

В таблице transactions учтите только строки с transaction_type = 'card_purchase'.

Напишите запрос, который для каждой категории вернёт merchant_category, сумма ставок и число транзакций в категории.

In [ ]:
SELECT 
    merchant_category
    ,SUM(amount) AS total_spent
    ,COUNT(transaction_id) AS num_purchases
FROM transactions
WHERE transaction_type = 'card_purchase'
GROUP BY merchant_category
ORDER BY total_spent DESC

#### #173 MAU — активные пользователи по месяцам

Пользователь считается активным в месяце, если он совершил хотя бы одну транзакцию любого типа в этом месяце.

Напишите запрос, который вернёт номер месяца и mau — число уникальных активных пользователей в этом месяце.

In [ ]:
SELECT 
    EXTRACT(MONTH FROM transaction_at) AS mth
    ,COUNT(DISTINCT customer_id) AS mau
FROM transactions
GROUP BY EXTRACT(MONTH FROM transaction_at)
ORDER BY mth

#### #176 Топ-10 покупателей по общей сумме покупок

Напишите запрос, который вернёт имя пользователя и суммарную сумму покупок клиента за всё время — для топ-10 покупателей. 

In [ ]:
SELECT 
    c.name AS customer_name
    ,SUM(amount) as total_spent
FROM transactions t
JOIN customers c ON c.customer_id = t.customer_id
GROUP BY c.name
ORDER BY total_spent DESC 
LIMIT 10

#### #179 Выручка по городам

Напишите запрос, который вернёт город и суммарную стоимость завершённых бронирований в этом городе. 

In [ ]:
SELECT 
    p.city
    ,SUM(total_price) as total_revenue
FROM bookings b
JOIN properties p ON p.property_id = b.property_id
WHERE b.status = 'completed'
GROUP BY p.city
ORDER BY total_revenue DESC  